In [0]:
file_path = "/Volumes/main/nyc-taxi/nyc_taxi_data/yellow_tripdata_2016-01.parquet"

df = spark.read.parquet(file_path)

display(df.limit(5))

In [0]:
from functools import reduce

base_path = "/Volumes/main/nyc-taxi/nyc_taxi_data"

files = [
    f"{base_path}/yellow_tripdata_2016-01.parquet",
    f"{base_path}/yellow_tripdata_2016-02.parquet",
    f"{base_path}/yellow_tripdata_2016-03.parquet",
]

dataframes = [spark.read.parquet(path) for path in files]

for path, df in zip(files, dataframes):
    print(f"\nFile: {path.split('/')[-1]}")
    print(f"Rows: {df.count():,}")
    print(f"Columns: {len(df.columns)}")
    df.printSchema()

# Check whether all files have the same columns and data types
schemas = [df.schema for df in dataframes]
assert all(schema == schemas[0] for schema in schemas), \
    "The files do not have identical schemas!"

print("\nAll three files have identical schemas.")

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

base_path = "/Volumes/main/nyc-taxi/nyc_taxi_data"

files = [
    f"{base_path}/yellow_tripdata_2016-01.parquet",
    f"{base_path}/yellow_tripdata_2016-02.parquet",
    f"{base_path}/yellow_tripdata_2016-03.parquet",
]

# Read each file and preserve its source filename
dataframes = []

for path in files:
    df = spark.read.parquet(path)
    df = df.withColumn("source_file", F.lit(path.split("/")[-1]))
    dataframes.append(df)

# Combine the three monthly datasets
combined_df = dataframes[0]

for df in dataframes[1:]:
    combined_df = combined_df.unionByName(df)

# Add ingestion metadata
combined_df = (
    combined_df
    .withColumn("ingestion_run_id", F.lit("initial_2016_ingestion"))
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("schema_version", F.lit("1.0"))
)

print(f"Combined columns: {len(combined_df.columns)}")
print(f"Source files: {len(files)}")


In [0]:
display(spark.table("main.`nyc-taxi`.bronze_taxi_trips").limit(10))

In [0]:
spark.sql("SHOW TABLES IN `main`.`nyc-taxi`").show()

In [0]:

from pyspark.sql import functions as F

# Hash the source file and original source columns
source_columns = [
    "VendorID",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "RatecodeID",
    "store_and_fwd_flag",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
]

bronze_df = combined_df.withColumn(
    "event_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("source_file"),
            *[
                F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
                for c in source_columns
            ],
        ),
        256,
    ),
)

target_path = "/Volumes/main/nyc-taxi/nyc_taxi_data/bronze_taxi_trips"

(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("main.`nyc-taxi`.bronze_taxi_trips")
)


In [0]:
bronze = spark.table("main.`nyc-taxi`.bronze_taxi_trips")

print(f"Rows: {bronze.count():,}")
print(f"Columns: {len(bronze.columns)}")

display(bronze.limit(10))

In [0]:
spark.table("main.`nyc-taxi`.bronze_taxi_trips").count()

In [0]:
bronze = spark.table("main.`nyc-taxi`.bronze_taxi_trips")

print("Rows:", f"{bronze.count():,}")
print("Columns:", len(bronze.columns))

bronze.printSchema()

display(bronze.select(
    "event_id",
    "source_file",
    "ingestion_run_id",
    "ingested_at",
    "schema_version"
).limit(10))

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("main.`nyc-taxi`.bronze_taxi_trips")

# 1. Verify row counts by source file
display(
    bronze.groupBy("source_file")
    .count()
    .orderBy("source_file")
)

# 2. Check missing event IDs and metadata
display(
    bronze.select(
        F.sum(F.col("event_id").isNull().cast("int")).alias("null_event_ids"),
        F.sum(F.col("source_file").isNull().cast("int")).alias("null_source_files"),
        F.sum(F.col("ingestion_run_id").isNull().cast("int")).alias("null_run_ids"),
        F.sum(F.col("ingested_at").isNull().cast("int")).alias("null_ingestion_timestamps")
    )
)

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("main.`nyc-taxi`.bronze_taxi_trips")

quality_summary = bronze.agg(
    F.sum(
        (F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"))
        .cast("int")
    ).alias("invalid_duration"),

    F.sum(
        (F.col("passenger_count") <= 0).cast("int")
    ).alias("invalid_passenger_count"),

    F.sum(
        (F.col("trip_distance") < 0).cast("int")
    ).alias("negative_distance"),

    F.sum(
        (F.col("tpep_pickup_datetime").isNull()
         | F.col("tpep_dropoff_datetime").isNull()).cast("int")
    ).alias("missing_timestamps")
)

display(quality_summary)

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("main.`nyc-taxi`.bronze_taxi_trips")

invalid_duration = (
    F.col("tpep_pickup_datetime").isNull()
    | F.col("tpep_dropoff_datetime").isNull()
    | (F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"))
)

invalid_passengers = (
    F.col("passenger_count").isNull()
    | (F.col("passenger_count") <= 0)
)

invalid_distance = (
    F.col("trip_distance").isNull()
    | (F.col("trip_distance") < 0)
)

invalid_condition = (
    invalid_duration
    | invalid_passengers
    | invalid_distance
)

invalid_trips = bronze.filter(invalid_condition)
valid_trips = bronze.filter(~invalid_condition)

print("Total Bronze:", bronze.count())
print("Valid trips:", valid_trips.count())
print("Invalid trips:", invalid_trips.count())

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("main.`nyc-taxi`.bronze_taxi_trips")

invalid_duration = (
    F.col("tpep_pickup_datetime").isNull()
    | F.col("tpep_dropoff_datetime").isNull()
    | (F.col("tpep_dropoff_datetime") <= F.col("tpep_pickup_datetime"))
)

invalid_passengers = (
    F.col("passenger_count").isNull()
    | (F.col("passenger_count") <= 0)
)

invalid_distance = (
    F.col("trip_distance").isNull()
    | (F.col("trip_distance") < 0)
)

invalid_condition = (
    invalid_duration | invalid_passengers | invalid_distance
)

valid_trips = bronze.filter(~invalid_condition)
invalid_trips = (
    bronze.filter(invalid_condition)
    .withColumn("quarantine_reason", F.concat_ws(
        ", ",
        F.when(invalid_duration, F.lit("invalid_duration")),
        F.when(invalid_passengers, F.lit("invalid_passenger_count")),
        F.when(invalid_distance, F.lit("invalid_distance"))
    ))
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_trips.write.format("delta").mode("overwrite").saveAsTable(
    "main.`nyc-taxi`.silver_taxi_trips"
)

invalid_trips.write.format("delta").mode("overwrite").saveAsTable(
    "main.`nyc-taxi`.quarantine_taxi_trips"
)

print("Silver rows:", spark.table("main.`nyc-taxi`.silver_taxi_trips").count())
print("Quarantine rows:", spark.table("main.`nyc-taxi`.quarantine_taxi_trips").count())